In [ ]:
import os
import math
import numpy as np
import xarray as xr
import scipy.ndimage
from pathlib import Path
import netCDF4 as nc
import cc3d
import gc
import sys

negative_w_threshold = -0.25
t=6

# Input
input_dir = Path("/mnt/stor-pool-01/projects/heus/ShellAnalysis/full-area")

In [ ]:
#Check that files exist
path_ds_ql_mask = Path(input_dir / "ql_mask.nc")
if not path_ds_ql_mask.is_file():
    print(f"❌ ERROR: Simulation dataset not found at:\n   {path_ds_ql_mask}\nEnding program early.", file=sys.stderr)
    sys.exit(1)
path_ds_w_mask = Path(input_dir / "w_mask.nc")
if not path_ds_w_mask.is_file():
    print(f"❌ ERROR: Simulation dataset not found at:\n   {path_ds_w_mask}\nEnding program early.", file=sys.stderr)
    sys.exit(1)

#Loading datasets
ds_ql_mask = xr.open_dataset(path_ds_ql_mask, decode_times=False)
ds_w_mask = xr.open_dataset(path_ds_w_mask, decode_times=False)

In [ ]:
#Making the shell
num_times = int(ds_ql_mask.time.size)
nz, ny, nx = ds_ql_mask.shell_mask.shape[1:]
z_coordinates = ds_ql_mask.z.values

#-Creating array for expansion
expansion = np.zeros((3,3,3), dtype=bool)
expansion[1, 1, :] = True  # X axis
expansion[1, :, 1] = True  # Y axis
expansion[:, 1, 1] = True  # Z axis

print(f"\n--- Processing Timestep {t}/{num_times - 1} ---")
#Slices
ql_raw = ds_ql_mask.ql_mask.isel(time=t).values.astype(bool)
w_slice = ds_w_mask.w_mask.isel(time=t).values.astype(bool)

local_shell_origin = np.full((nz,ny,nx), np.nan, dtype=np.float32)
local_shell_termination = np.full((nz,ny,nx), np.nan, dtype=np.float32)

local_shell_cloud_bottom = np.full((nz,ny,nx), np.nan, dtype=np.float32)
local_shell_cloud_top = np.full((nz,ny,nx), np.nan, dtype=np.float32)

local_congestus_mask = np.full((nz,ny,nx), False, dtype=bool)
local_deep_mask = np.full((nz,ny,nx), False, dtype=bool)

#Step 1 - Recreating the shell labels
# Dilated ql serving working space for shell expansion and intersection detection
padded_ql = np.pad(ql_raw, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0) #padded z with 0's to prevent 
padded_ql = np.pad(padded_ql, ((0, 0), (1, 1), (1, 1)), mode='wrap')
padded_current = scipy.ndimage.binary_dilation(padded_ql, structure=expansion)
current = padded_current[1:-1, 1:-1, 1:-1]

padded_w = np.pad(w_slice, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0)
padded_labels = cc3d.connected_components(padded_w, connectivity=6, periodic_boundary=True) #labels every region in w
labels = padded_labels[1:-1, :, :]

num_features = np.max(labels)
if num_features == 0:
    print(f"No objects found in timestep {t}. Writing empty layer matrix.")
    sys.exit(0)

#-get w regions that intersect with ql
matching_labels = set(labels[current])
matching_labels.discard(0)

matching_labels_ql = set(labels[ql_raw])
matching_labels_ql.discard(0)

shell_origin_dict       = {}
shell_termination_dict  = {}
cloud_bottom_dict       = {}
cloud_top_dict          = {}

if not matching_labels:
    print(f"No valid convective features connected to cloud in timestep {t}.")
    sys.exit(0)

#Shell Depth
valid_shell_labels = np.where(current, labels, 0)
valid_ql_labels = np.where(ql_raw, labels, 0)
slices = scipy.ndimage.find_objects(valid_shell_labels)
slices_ql = scipy.ndimage.find_objects(valid_ql_labels)

for obj_id in matching_labels:
    obj_slice = slices[obj_id - 1] if obj_id - 1 < len(slices) else None

    if obj_slice is not None:
        z_slice = obj_slice[0]
        min_z_phys = z_coordinates[z_slice.start]
        max_z_phys = z_coordinates[z_slice.stop - 1]

        box_mask = valid_shell_labels[obj_slice] == obj_id

        local_shell_origin[obj_slice][box_mask] = min_z_phys
        local_shell_termination[obj_slice][box_mask] = max_z_phys

        shell_origin_dict[obj_id - 1] = min_z_phys
        shell_termination_dict[obj_id - 1] = max_z_phys

        #cloud min and max
        ql_obj_slice = slices_ql[obj_id - 1] if obj_id - 1 < len(slices_ql) else None
        if ql_obj_slice is not None:
            z_ql_slice = ql_obj_slice[0]
            min_z_ql = z_coordinates[z_ql_slice.start]
            max_z_ql = z_coordinates[z_ql_slice.stop - 1]
            
            if(max_z_ql > 5000):
                local_deep_mask[obj_slice][box_mask] = True
            elif(max_z_ql > 2000):
                local_congestus_mask[obj_slice][box_mask] = True

            local_shell_cloud_bottom[obj_slice][box_mask] = min_z_ql
            local_shell_cloud_top[obj_slice][box_mask] = max_z_ql

            cloud_bottom_dict[obj_id]  = min_z_ql
            cloud_top_dict[obj_id]     = max_z_ql
    



#Export list
#- shell origin and termination
#- shell origin list and termination list
#- cloud bottom and top
#- cloud bottom list and cloud top list
#- congestus mask
#- deep mask
#- cloud labels (matching_labels)
#- ql labels ()
print(f"Shell depths constructed in timestep {t}.")
